In [1]:
# %%
"""
CELL 1 — SETUP
enhancement stat bonuses + value @ item level 60
"""
import json
import re
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import pandas as pd

BASE = Path("..")
RAW = BASE / "data" / "ncs_raw_json"
DICT_PATH = BASE / "data" / "dictionary" / "result" / "dictionary_en_ru.json"
OUT_DIR = BASE / "data" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ITEM_LEVEL = 60  # inventory_experience_level

GUID_TRIPLE = re.compile(
    r"([A-Za-z0-9_.'/\-]{2,120}),\s*([0-9A-Fa-f]{32}),\s*([^\n\r\"]{2,200})"
)

def jload(p: Path):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def clean(s: str) -> str:
    s = re.sub(r"\{[^}]*\}", "", s)
    s = re.sub(r"\[/?[^\]]*\]", "", s)
    s = re.sub(r"\s+", " ", s).strip(" -:\n\t")
    return s

print("DICT:", DICT_PATH.resolve(), "exists:", DICT_PATH.exists())
print("ITEM_LEVEL =", ITEM_LEVEL)

DICT: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/dictionary/result/dictionary_en_ru.json exists: True
ITEM_LEVEL = 60


In [2]:
# %%
"""
CELL 2 — DICTIONARY
"""
with open(DICT_PATH, encoding="utf-8") as f:
    raw_dict = json.load(f)

guid_map = {}
en_map = defaultdict(list)

for k, v in raw_dict.items():
    if not isinstance(v, dict):
        continue
    en = (v.get("en") or "").strip()
    ru = (v.get("ru") or "").strip()
    entry = {"en": en, "ru": ru}
    ku = str(k).replace("-", "").upper()
    guid_map[ku] = entry
    guid_map[str(k).lower()] = entry
    if en:
        en_map[en.lower()].append(ru if ru else None)

def translate_by_guid(guid: str | None) -> str:
    if not guid:
        return "—"
    g = str(guid).replace("-", "").upper()
    hit = guid_map.get(g) or guid_map.get(str(guid).lower())
    if not hit:
        return "(перевод не найден)"
    ru = (hit.get("ru") or "").strip()
    return ru if ru else "(перевод не найден)"

def translate_by_en(text: str | None) -> str:
    if not text or text == "—":
        return "—"
    variants = [r for r in en_map.get(text.lower(), []) if r]
    if not variants:
        return "(перевод не найден)"
    uniq = list(dict.fromkeys(variants))
    if len(uniq) > 1:
        return "(требуется ручная проверка)"
    return uniq[0]

print(f"guid_map={len(guid_map)}")

guid_map=232656


In [3]:
# %%
"""
CELL 3 — SOURCE A: part_stat keys from inv *enhancement*
         SOURCE B: uistat_enh_stat_* labels + GUID
         JOIN by suffix
"""
# --- A ---
stat_keys = set()
for path in sorted(p for p in RAW.rglob("inv*.json") if "name" not in p.name.lower()):
    try:
        data = jload(path)
    except Exception:
        continue
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                it = str(entry.get("key") or "").lower()
                if "enhancement" not in it:
                    continue
                for dep in entry.get("dep_entries") or []:
                    dtn = str(dep.get("dep_table_name") or "").lower()
                    if dtn not in ("stat_group1", "stat_group2", "stat_group3"):
                        continue
                    dkey = str(dep.get("key") or "").lower()
                    if dkey.startswith("part_stat"):
                        stat_keys.add(dkey)
                    val = dep.get("value") if isinstance(dep.get("value"), dict) else {}
                    pairs = ((val.get("parttypeselectionrules") or {}).get("pairs") or {})
                    if not isinstance(pairs, dict):
                        continue
                    for pair in pairs.values():
                        if not isinstance(pair, dict):
                            continue
                        if str(pair.get("key") or "").lower() not in (
                            "stat_group1", "stat_group2", "stat_group3"
                        ):
                            continue
                        for p in (pair.get("value") or {}).get("parts") or []:
                            part = p.get("part") if isinstance(p, dict) else p
                            if part and str(part).lower().startswith("part_stat"):
                                stat_keys.add(str(part).lower())

print(f"part_stat keys: {len(stat_keys)}")

# --- B ---
label_by_suffix = {}
for path in sorted(RAW.rglob("*.json")):
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    if "uistat_enh_stat_" not in text.lower() and "enhancement_uistats" not in text.lower():
        continue
    for m in GUID_TRIPLE.finditer(text):
        cat, guid, disp = m.group(1).strip(), m.group(2).upper(), m.group(3).strip()
        name = clean(disp)
        if not name or len(name) < 2 or len(name) > 100:
            continue
        win = text[max(0, m.start() - 300): m.start() + 80]
        keys = re.findall(r"uistat_enh_stat_[a-z0-9_]+", win, re.I)
        if not keys and "enhancement_uistats" not in cat.lower():
            continue
        uk = keys[-1].lower() if keys else None
        if not uk:
            continue
        suf = uk.replace("uistat_enh_stat_", "")
        label_by_suffix[suf] = {"name_eng": name, "guid": guid, "uistat_key": uk}

print(f"uistat labels: {len(label_by_suffix)}")

def suffixes_from_part(pk: str) -> list[str]:
    p = pk.lower()
    for pref in ("part_stat3_", "part_stat2_", "part_stat_"):
        if p.startswith(pref):
            return [p[len(pref):]]
    return []

# name_lower → {name_eng, guid, part_suffixes}
uniq = {}
for pk in sorted(stat_keys):
    hit = None
    sufs = suffixes_from_part(pk)
    for suf in sufs:
        hit = label_by_suffix.get(suf)
        if hit:
            break
        for k, v in label_by_suffix.items():
            if k == suf or k.endswith(suf) or suf.endswith(k):
                hit = v
                break
        if hit:
            break
    if not hit:
        continue
    n = re.sub(r"\s+is increased by\s*$", "", hit["name_eng"], flags=re.I).strip()
    sig = n.lower()
    if sig not in uniq:
        uniq[sig] = {
            "name_eng": n,
            "guid": hit["guid"],
            "suffixes": set(sufs),
        }
    else:
        uniq[sig]["suffixes"].update(sufs)

print(f"unique bonuses: {len(uniq)}")

part_stat keys: 216
uistat labels: 72
unique bonuses: 62


In [4]:
# %%
"""
CELL 4 — VALUES from NCS only (no hardcoded stat name map)
Chain:
  part_stat → Enh_Stat_*_Modifier (from inv aspects)
  Enh_Stat_* → DataTable_Enhancement_Modifiers.rowname (from attribute*)
  rowname → modifier float
  value@ilvl = modifier * (1 + formula_b_scalar * ilvl)
"""
from collections import defaultdict

# ---------------------------------------------------------------------------
# 4a) part_key → attribute name (Enh_Stat_...)
# ---------------------------------------------------------------------------
part_to_attr: dict[str, str] = {}

for path in sorted(p for p in RAW.rglob("inv*.json") if "name" not in p.name.lower()):
    try:
        data = jload(path)
    except Exception:
        continue
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                for dep in entry.get("dep_entries") or []:
                    pk = str(dep.get("key") or "").lower()
                    if not pk.startswith("part_stat"):
                        continue
                    val = dep.get("value") if isinstance(dep.get("value"), dict) else {}
                    aspects = val.get("aspects") or []
                    if not isinstance(aspects, list):
                        continue
                    for asp in aspects:
                        if not isinstance(asp, dict):
                            continue
                        aug = asp.get("augment") or {}
                        if not isinstance(aug, dict):
                            continue
                        for eff in aug.get("modifierattributeeffects") or []:
                            if not isinstance(eff, dict):
                                continue
                            mv = eff.get("modifiervalue") or {}
                            if not isinstance(mv, dict):
                                continue
                            attr = str(mv.get("attribute") or "")
                            # attribute'Enh_Stat_Weapon_Damage_Modifier'
                            m = re.search(
                                r"Enh_Stat_[A-Za-z0-9_]+",
                                attr,
                                re.I,
                            )
                            if m:
                                part_to_attr[pk] = m.group(0)

print(f"part→attr links: {len(part_to_attr)}")
for k, v in sorted(part_to_attr.items())[:8]:
    print(f"  {k} → {v}")

# ---------------------------------------------------------------------------
# 4b) Enh_Stat_* attribute → (table, rowname, column)
# ---------------------------------------------------------------------------
attr_to_row: dict[str, dict] = {}

for path in sorted(RAW.rglob("attribute*.json")):
    try:
        data = jload(path)
    except Exception:
        continue

    def walk(o):
        if isinstance(o, dict):
            k = str(o.get("key") or "").lower()
            if k.startswith("enh_stat_") and k.endswith("_modifier"):
                blob = json.dumps(o, ensure_ascii=False)
                # rowname + datatable + columnname near this attr
                # prefer DataTable_Enhancement_Modifiers
                rn = None
                col = None
                tbl = None
                for m in re.finditer(
                    r'"rowname"\s*:\s*"([^"]+)"',
                    blob,
                    re.I,
                ):
                    cand = m.group(1)
                    if cand.lower().startswith("stat_"):
                        rn = cand
                        break
                if not rn:
                    for m in re.finditer(
                        r'"rowname"\s*:\s*"([^"]+)"',
                        blob,
                        re.I,
                    ):
                        rn = m.group(1)
                        break
                cm = re.search(r'"columnname"\s*:\s*"([^"]+)"', blob, re.I)
                if cm:
                    col = cm.group(1)
                tm = re.search(
                    r"DataTable_Enhancement_Modifiers|datatable_enhancement_modifiers",
                    blob,
                    re.I,
                )
                if tm:
                    tbl = "DataTable_Enhancement_Modifiers"
                if rn:
                    attr_canon = None
                    am = re.search(r"Enh_Stat_[A-Za-z0-9_]+", blob)
                    if am:
                        attr_canon = am.group(0)
                    # key itself
                    km = re.search(r"enh_stat_[a-z0-9_]+", k)
                    if not attr_canon and km:
                        # title-ish
                        parts = km.group(0).split("_")
                        attr_canon = "_".join(
                            p.capitalize() if i > 0 else p.capitalize()
                            for i, p in enumerate(parts)
                        )
                        # better: from attribute field
                    am2 = re.search(
                        r'"attribute"\s*:\s*"([^"]*Enh_Stat_[^"]+)"',
                        blob,
                    )
                    if am2:
                        attr_canon = re.search(
                            r"Enh_Stat_[A-Za-z0-9_]+", am2.group(1)
                        )
                        attr_canon = attr_canon.group(0) if attr_canon else am2.group(1)
                    if attr_canon:
                        prev = attr_to_row.get(attr_canon)
                        # prefer rows that look like Stat_*
                        if prev is None or (
                            rn.lower().startswith("stat_")
                            and not str(prev.get("rowname") or "").lower().startswith("stat_")
                        ):
                            attr_to_row[attr_canon] = {
                                "rowname": rn,
                                "column": col or "Modifier",
                                "table": tbl,
                                "file": path.name,
                            }
            for v in o.values():
                walk(v)
        elif isinstance(o, list):
            for i in o:
                walk(i)

    walk(data)

print(f"\nattr→row links: {len(attr_to_row)}")
for k, v in sorted(attr_to_row.items()):
    print(f"  {k} → {v['rowname']} ({v.get('table')})")

# ---------------------------------------------------------------------------
# 4c) rowname → modifier float (all Stat_* from table)
# ---------------------------------------------------------------------------
table_mods: dict[str, float] = {}

for path in sorted(RAW.rglob("gbx_ue_data_table*.json")):
    try:
        data = jload(path)
    except Exception:
        continue

    def walk(o):
        if isinstance(o, dict):
            key = str(o.get("key") or "").lower()
            val = o.get("value") if isinstance(o.get("value"), dict) else {}
            tname = str(val.get("gbx_ue_data_table") or key)
            if "enhancement_modifier" in tname.lower() or key == "datatable_enhancement_modifiers":
                for row in val.get("data") or []:
                    if not isinstance(row, dict):
                        continue
                    rn = str(row.get("row_name") or "")
                    rv = row.get("row_value") or {}
                    if isinstance(rv, dict) and "modifier" in rv:
                        try:
                            table_mods[rn] = float(rv["modifier"])
                        except (TypeError, ValueError):
                            pass
            rn = str(o.get("row_name") or "")
            rv = o.get("row_value") if isinstance(o.get("row_value"), dict) else {}
            if rn and isinstance(rv, dict) and "modifier" in rv:
                if rn not in table_mods:
                    try:
                        table_mods[rn] = float(rv["modifier"])
                    except (TypeError, ValueError):
                        pass
            for v in o.values():
                walk(v)
        elif isinstance(o, list):
            for i in o[:10000]:
                walk(i)

    walk(data)

print(f"\ntable modifiers: {len(table_mods)}")
for k, v in sorted(table_mods.items()):
    if k.lower().startswith("stat_"):
        print(f"  {k}: {v}")

# ---------------------------------------------------------------------------
# 4d) scalar 0.08 from Enh_Stat_Modifier_Formula_B (not hardcoded if found)
# ---------------------------------------------------------------------------
formula_scalar = None
for path in sorted(RAW.rglob("attribute*.json")):
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    if "enh_stat_modifier_formula_b" not in text.lower():
        continue
    # window around formula_b
    for m in re.finditer(
        r"enh_stat_modifier_formula_b.{0,800}",
        text,
        re.I | re.S,
    ):
        win = m.group(0)
        if "inventory_experience_level" not in win.lower():
            continue
        sm = re.search(
            r'"scalar"\s*:\s*\{[^}]*"constant"\s*:\s*"([0-9.]+)"',
            win,
            re.I | re.S,
        )
        if sm:
            formula_scalar = float(sm.group(1))
            break
    if formula_scalar is not None:
        break

if formula_scalar is None:
    # fallback only if attribute dump incomplete — still print warning
    formula_scalar = 0.08
    print("WARN: formula scalar not found in NCS, using 0.08 from prior resolve")
else:
    print(f"formula scalar from NCS: {formula_scalar}")

SCALE = 1.0 + formula_scalar * float(ITEM_LEVEL)
print(f"SCALE @ ilvl {ITEM_LEVEL} = {SCALE}")

# ---------------------------------------------------------------------------
# 4e) helpers: part suffixes / part keys → base modifier
# ---------------------------------------------------------------------------
def base_for_part_keys(part_keys: set[str]) -> float | None:
    """Follow part → attr → row → modifier. No name whitelist."""
    for pk in part_keys:
        attr = part_to_attr.get(pk)
        if not attr:
            # try without part_stat2_/3_ normalize
            continue
        info = attr_to_row.get(attr)
        if not info:
            # case-insensitive attr match
            for ak, av in attr_to_row.items():
                if ak.lower() == attr.lower():
                    info = av
                    break
        if not info:
            continue
        rn = info["rowname"]
        if rn in table_mods:
            return table_mods[rn]
        # case-insensitive row
        for tk, tv in table_mods.items():
            if tk.lower() == rn.lower():
                return tv
    return None

def format_value_pct(base: float | None) -> tuple[str, str]:
    if base is None:
        return "—", "—"
    raw = base * SCALE
    pct = abs(raw) * 100.0
    if abs(pct - round(pct)) < 0.05:
        txt = f"+{int(round(pct))}%"
    else:
        txt = f"+{pct:.1f}%"
    return txt, f"{raw:.6f}"

print("CELL 4 OK")

part→attr links: 198
  part_stat2_statuseffect_chance → Enh_Stat_StatusEffect_Chance_Modifier
  part_stat2_statuseffect_damage → Enh_Stat_StatusEffect_Damage_Modifier
  part_stat2_weapon_accuracy → Enh_Stat_Weapon_Accuracy_Modifier
  part_stat2_weapon_ads_proficiency → Enh_Stat_Weapon_ADS_Proficiency_Modifier
  part_stat2_weapon_criticaldamage → Enh_Stat_Weapon_CriticalDamage_Modifier
  part_stat2_weapon_damage → Enh_Stat_Weapon_Damage_Modifier
  part_stat2_weapon_equipspeed → Enh_Stat_Weapon_EquipSpeed_Modifier
  part_stat2_weapon_firerate → Enh_Stat_Weapon_FireRate_InverseModifier

attr→row links: 27
  Enh_Stat_Modifier_Formula → Stat_StatusEffect_Chance (DataTable_Enhancement_Modifiers)
  Enh_Stat_StatusEffect_Chance_Modifier → Stat_StatusEffect_Chance (DataTable_Enhancement_Modifiers)
  Enh_Stat_StatusEffect_Damage_Modifier → Stat_StatusEffect_Damage (DataTable_Enhancement_Modifiers)
  Enh_Stat_WT_StatusEffect_Chance_Modifier → Stat_StatusEffect_Chance (DataTable_Enhancement_Modifi

In [5]:
# %%
"""
CELL 5 — BUILD DF + SAVE
Also rebuild part_keys per display name from cell 3 uniq suffixes → actual keys
"""
# reverse: suffix → part keys we saw
suf_to_parts: dict[str, set[str]] = defaultdict(set)
for pk in stat_keys:
    for pref in ("part_stat3_", "part_stat2_", "part_stat_"):
        if pk.startswith(pref):
            suf_to_parts[pk[len(pref):]].add(pk)
            break

rows = []
for v in sorted(uniq.values(), key=lambda x: x["name_eng"].lower()):
    eng = v["name_eng"]
    ru = translate_by_guid(v["guid"])
    if ru in ("(перевод не найден)", "—"):
        ru = translate_by_en(eng)

    part_keys = set()
    for suf in v["suffixes"]:
        part_keys |= suf_to_parts.get(suf, set())
    # also try full keys that end with suffix
    for pk in stat_keys:
        for suf in v["suffixes"]:
            if pk.endswith(suf):
                part_keys.add(pk)

    base = base_for_part_keys(part_keys)
    val_txt, val_raw = format_value_pct(base)

    rows.append({
        "name_eng": eng,
        "name_ru": ru,
        "modifier_base": f"{base:.6f}" if base is not None else "—",
        "value_at_ilvl_60": val_txt,
        "value_at_ilvl_60_raw": val_raw,
    })

df = pd.DataFrame(rows).drop_duplicates(subset=["name_eng"]).reset_index(drop=True)
print(df.shape)
print(df.to_string())

ts = datetime.now().strftime("%m-%d-%Y_%I-%M-%S%p")
out = OUT_DIR / f"bl4_enhancement_stat_bonuses_{ts}.csv"
df.to_csv(out, index=False, encoding="utf-8-sig", sep=";", quoting=1)
print("\nWrote:", out.resolve())
print(f"with value @60: {(df['value_at_ilvl_60'] != '—').sum()} / {len(df)}")

(62, 5)
                         name_eng                                                                                                            name_ru modifier_base value_at_ilvl_60 value_at_ilvl_60_raw
0          Assault Rifle Accuracy                               [secondary]Точность автоматов[/secondary] повышается на [secondary]{mod}[/secondary]     -0.100000             +58%            -0.580000
1   Assault Rifle ADS Proficiency                   [secondary]Навык прицеливания из автомата[/secondary] повышается на [secondary]{mod}[/secondary]     -0.100000             +58%            -0.580000
2   Assault Rifle Critical Damage                    [secondary]Критический урон от автоматов[/secondary] повышается на [secondary]{mod}[/secondary]      0.100000             +58%             0.580000
3            Assault Rifle Damage                                [secondary]Урон от автоматов[/secondary] повышается на [secondary]{mod}[/secondary]      0.050000             +29%         